# Assignment 6 – Build an Interactive Dashboard with Plotly Dash

This notebook contains the completed Python implementation for the four tasks in the lab PDF:
1. Launch Site dropdown
2. Success/failure pie chart callback
3. Payload range slider (0–10,000 kg)
4. Payload vs. launch outcome scatter plot callback

**Environment:** Skills Network Cloud IDE / Python 3.11. Keep `spacex_launch_dash.csv` in the same directory as this notebook.

Before running, install the packages in a terminal:
```bash
python3.11 -m pip install pandas dash plotly
```
Download the dataset if needed:
```bash
wget "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/spacex_launch_dash.csv"
```
Dash starts a web server on port **8050**. Run the final cell, then use the Cloud IDE's **Launch Application** option for port 8050.

## 1. Import libraries and load the dataset

In [11]:
!pip install pandas dash plotly

In [12]:
!wget "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/spacex_launch_dash.csv"

--2026-11-01 11:50:24--  https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/spacex_launch_dash.csv
Resolving cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud (cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud)... 169.63.118.104
Connecting to cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud (cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud)|169.63.118.104|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2476 (2.4K) [text/csv]
Saving to: ‘spacex_launch_dash.csv.2’

spacex_launch_dash. 100%[===================>]   2.42K  --.-KB/s    in 0s      

2026-11-01 11:50:24 (894 MB/s) - ‘spacex_launch_dash.csv.2’ saved [2476/2476]



In [13]:
from pathlib import Path
import pandas as pd
import plotly.express as px
from dash import Dash, dcc, html
from dash.dependencies import Input, Output


In [14]:
DATA_FILE = Path('spacex_launch_dash.csv')

if not DATA_FILE.exists():
    raise FileNotFoundError(
        'spacex_launch_dash.csv was not found. Download it and place it in the same folder as this notebook.'
    )

spacex_df = pd.read_csv(DATA_FILE)
spacex_df.head()


,Unnamed: 0,Flight Number,Launch Site,class,Payload Mass (kg),Booster Version,Booster Version Category
0,0,1,CCAFS LC-40,0,0.0,F9 v1.0 B0003,v1.0
1,1,2,CCAFS LC-40,0,0.0,F9 v1.0 B0004,v1.0
2,2,3,CCAFS LC-40,0,525.0,F9 v1.0 B0005,v1.0
3,3,4,CCAFS LC-40,0,500.0,F9 v1.0 B0006,v1.0
4,4,5,CCAFS LC-40,0,677.0,F9 v1.0 B0007,v1.0


## 2. Prepare the data and launch-site options

In [15]:
required_columns = {
    'Launch Site',
    'class',
    'Payload Mass (kg)',
    'Booster Version Category',
}
missing_columns = required_columns.difference(spacex_df.columns)
if missing_columns:
    raise ValueError('The CSV is missing required columns: ' + ', '.join(sorted(missing_columns)))

spacex_df['class'] = pd.to_numeric(spacex_df['class'], errors='coerce')
spacex_df['Payload Mass (kg)'] = pd.to_numeric(spacex_df['Payload Mass (kg)'], errors='coerce')

min_payload = 0
max_payload = 10000
launch_sites = sorted(spacex_df['Launch Site'].dropna().astype(str).unique().tolist())
print('Launch sites:', launch_sites)
print('Dataset shape:', spacex_df.shape)


Launch sites: ['CCAFS LC-40', 'CCAFS SLC-40', 'KSC LC-39A', 'VAFB SLC-4E']
Dataset shape: (56, 7)


## 3. Initialize the Dash application and create the dashboard layout

In [16]:
app = Dash(__name__)
app.title = 'SpaceX Launch Records Dashboard'

app.layout = html.Div(
    children=[
        html.H1(
            'SpaceX Launch Records Dashboard',
            style={'textAlign': 'center', 'color': '#503D36', 'fontSize': '40px'},
        ),
        html.Div(
            children=[
                html.Label('Select Launch Site:'),
                dcc.Dropdown(
                    id='site-dropdown',
                    options=[{'label': 'All Sites', 'value': 'ALL'}]
                    + [{'label': site, 'value': site} for site in launch_sites],
                    value='ALL',
                    placeholder='Select a Launch Site here',
                    searchable=True,
                    clearable=False,
                ),
            ],
            style={'width': '80%', 'margin': 'auto'},
        ),
        html.Br(),
        dcc.Graph(id='success-pie-chart'),
        html.Br(),
        html.Div(
            children=[
                html.Label('Payload Range (Kg):'),
                dcc.RangeSlider(
                    id='payload-slider',
                    min=min_payload,
                    max=max_payload,
                    step=1000,
                    marks={value: str(value) for value in range(0, max_payload + 1, 1000)},
                    value=[min_payload, max_payload],
                    tooltip={'placement': 'bottom', 'always_visible': True},
                ),
            ],
            style={'width': '80%', 'margin': 'auto'},
        ),
        html.Br(),
        dcc.Graph(id='success-payload-scatter-chart'),
    ],
    style={'fontFamily': 'Arial, sans-serif', 'padding': '20px'},
)

print('Dashboard layout created.')


Dashboard layout created.


## 4. TASK 2 – Callback for the success/failure pie chart

In [17]:
@app.callback(
    Output('success-pie-chart', 'figure'),
    Input('site-dropdown', 'value'),
)
def get_pie_chart(entered_site):
    if entered_site == 'ALL':
        success_by_site = (
            spacex_df[spacex_df['class'] == 1]
            .groupby('Launch Site')
            .size()
            .reset_index(name='Success Count')
        )
        if success_by_site.empty:
            return px.pie(title='No successful launches found in the dataset')
        return px.pie(
            success_by_site,
            values='Success Count',
            names='Launch Site',
            title='Total Successful Launches by Launch Site',
        )

    filtered_df = spacex_df[spacex_df['Launch Site'] == entered_site]
    success_count = int((filtered_df['class'] == 1).sum())
    failure_count = int((filtered_df['class'] == 0).sum())
    outcome_df = pd.DataFrame({'Outcome': ['Success', 'Failure'], 'Count': [success_count, failure_count]})
    return px.pie(
        outcome_df,
        values='Count',
        names='Outcome',
        title=f'Launch Outcomes for {entered_site}',
    )

print('Pie-chart callback registered.')


Pie-chart callback registered.


## 5. TASK 4 – Callback for payload vs. launch outcome scatter plot

In [18]:
@app.callback(
    Output('success-payload-scatter-chart', 'figure'),
    [
        Input('site-dropdown', 'value'),
        Input('payload-slider', 'value'),
    ],
)
def get_scatter_plot(entered_site, payload_range):
    low, high = payload_range
    filtered_df = spacex_df[
        spacex_df['Payload Mass (kg)'].between(low, high, inclusive='both')
    ].copy()

    if entered_site != 'ALL':
        filtered_df = filtered_df[filtered_df['Launch Site'] == entered_site]

    if filtered_df.empty:
        return px.scatter(title='No launches found for the selected filters')

    fig = px.scatter(
        filtered_df,
        x='Payload Mass (kg)',
        y='class',
        color='Booster Version Category',
        title='Payload Mass vs. Launch Outcome',
        labels={
            'class': 'Launch Outcome (0 = Failure, 1 = Success)',
            'Payload Mass (kg)': 'Payload Mass (kg)',
            'Booster Version Category': 'Booster Version',
        },
        hover_data=['Launch Site'],
    )
    fig.update_yaxes(tickmode='array', tickvals=[0, 1], ticktext=['Failure', 'Success'])
    return fig

print('Scatter-plot callback registered.')


Scatter-plot callback registered.


## 6. Run the dashboard

Run this final cell. The Dash server will start on port **8050**. In the Cloud IDE, use **Others → Launch Application**, enter `8050`, and open the application.

**Note:** Keep this cell running while using the dashboard. Stop the server with the notebook interrupt/stop control when finished.

In [19]:
app.run(host='0.0.0.0', port=8050, debug=False)


<IPython.core.display.Javascript object>